## Configuração

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (FunctionTransformer, MinMaxScaler, OneHotEncoder,
                                   OrdinalEncoder, StandardScaler)
from sklearn.utils.class_weight import compute_class_weight

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
ARQUIVO_BRUTO = RAIZ / "data" / "raw" / "online_shoppers_intention.csv"
ARQUIVO_PROCESSADO = RAIZ / "data" / "processed" / "online_shoppers_limpo.csv"
DIR_PIPELINES = RAIZ / "02_preprocessamento" / "pipelines"
DIR_PIPELINES.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
pd.set_option("display.max_columns", 60)

## 1. Etapas comuns a todos os algoritmos

| O que é feito | Justificativa |
|---|---|
| Remover duplicatas | A EDA encontrou 125 registros duplicados. Com o k-fold, uma cópia poderia cair no treino e outra no teste, e o modelo seria avaliado numa instância que já viu. |
| Padronizar `June` → `Jun` | Na EDA, junho era o único mês fora do padrão de três letras. |
| `Weekend` e `Revenue` como 0/1 | Os algoritmos do scikit-learn trabalham com números; booleanos viram 0 (False) e 1 (True). |

Nenhuma dessas etapas aprende parâmetros com os dados, então podem ser aplicadas antes da divisão em folds.

In [2]:
bruto = pd.read_csv(ARQUIVO_BRUTO)
df = bruto.drop_duplicates().reset_index(drop=True)
df["Month"] = df["Month"].replace({"June": "Jun"})
df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)

comparacao = pd.DataFrame({
    "instancias": [len(bruto), len(df)],
    "nao_comprou": [int((~bruto["Revenue"]).sum()), int((df["Revenue"] == 0).sum())],
    "comprou": [int(bruto["Revenue"].sum()), int(df["Revenue"].sum())],
}, index=["base original", "após etapas comuns"])
comparacao["pct_comprou"] = 100 * comparacao["comprou"] / comparacao["instancias"]
print(f"Duplicatas removidas: {len(bruto) - len(df)}")
print("Meses:", sorted(df["Month"].unique()))
comparacao

Duplicatas removidas: 125
Meses: ['Aug', 'Dec', 'Feb', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep']


,instancias,nao_comprou,comprou,pct_comprou
base original,12330,10422,1908,15.474453
após etapas comuns,12205,10297,1908,15.632937


In [3]:
df.to_csv(ARQUIVO_PROCESSADO, index=False)
print(f"Base salva em {ARQUIVO_PROCESSADO.relative_to(RAIZ)} ({df.shape[0]} linhas x {df.shape[1]} colunas)")
df.head()

Base salva em data/processed/online_shoppers_limpo.csv (12205 linhas x 18 colunas)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,1,0


## 2. Grupos de atributos usados pelos pipelines

- **Redundantes (removidos no KNN e no MLP):** em cada par contagem × duração, que têm correlação de Spearman entre 0,88
  e 0,95, mantém-se a **duração** e remove-se a contagem. A duração é mais granular (milhares de valores distintos,
  contra algumas dezenas na contagem) e já reflete, em boa parte, quantas páginas foram vistas.
- **Categóricos:** `Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType` e `VisitorType`. Os quatro códigos
  numéricos são categorias: o Browser 13 não é "maior" que o Browser 2.
- **`Weekend`:** binário (0/1), entra direto em todos os pipelines.

In [4]:
X = df.drop(columns="Revenue")
y = df["Revenue"]

CATEGORICOS = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType"]
REDUNDANTES = ["Administrative", "Informational", "ProductRelated"]
DURACOES_E_PAGEVALUES = ["Administrative_Duration", "Informational_Duration",
                         "ProductRelated_Duration", "PageValues"]
TAXAS = ["BounceRates", "ExitRates", "SpecialDay"]
BINARIOS = ["Weekend"]
ORDEM_MESES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def one_hot_com_raras():
    """One-hot que agrupa categorias com menos de 1% das instâncias de treino numa coluna 'infrequent'."""
    return OneHotEncoder(min_frequency=0.01, handle_unknown="infrequent_if_exist", sparse_output=False)

## 3. Pipelines específicos de cada algoritmo

### 3.1 KNN

| O que é feito | Justificativa |
|---|---|
| Remover `Administrative`, `Informational` e `ProductRelated` | Correlação clara com as respectivas durações; no KNN cada par redundante pesaria em dobro na distância. |
| log(1 + x) nas durações e em `PageValues` | Atributos muito assimétricos: sem o log, cerca de 90% das sessões ficam espremidas em menos de 5% do intervalo após o min-max, e esses atributos quase não contam na distância. |
| Min-max em todos os atributos numéricos | Deixa os atributos na mesma escala [0, 1], para nenhum dominar a distância por ter valores maiores. |
| One-hot nos categóricos | A métrica de distância só lida com números, e usar os códigos diretamente criaria uma ordem que não existe. |
| Agrupar categorias com menos de 1% das sessões | Reduz o ruído e o número de colunas: categorias com 1 a 10 sessões não têm exemplos suficientes para ajudar. |
| Sobreamostragem aleatória da classe "comprou" (só no fold de treino) | O KNN não aceita pesos por classe e, com 84% de "não comprou", os vizinhos mais próximos tendem a ser dessa classe. A sobreamostragem aleatória replica instâncias reais e não cria valores sem sentido nas colunas one-hot, como o SMOTE comum faria. |

In [5]:
preprocessamento_knn = ColumnTransformer([
    ("log_minmax", make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                                 MinMaxScaler()), DURACOES_E_PAGEVALUES),
    ("minmax", MinMaxScaler(), TAXAS),
    ("onehot", one_hot_com_raras(), CATEGORICOS),
    ("binarios", "passthrough", BINARIOS),
], remainder="drop", verbose_feature_names_out=False)   # remainder="drop" remove os redundantes

pipeline_knn = Pipeline([
    ("preprocessamento", preprocessamento_knn),
    ("balanceamento", RandomOverSampler(random_state=SEMENTE)),
    ("modelo", KNeighborsClassifier()),
])
pipeline_knn

,steps,"[('preprocessamento', ...), ('balanceamento', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log_minmax', ...), ('minmax', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more d

### 3.2 Random Forest

| O que é feito | Justificativa |
|---|---|
| Manter todos os atributos | A correlação entre atributos não atrapalha a predição das árvores; só divide a importância entre os pares. |
| Sem log e sem escala | As árvores dividem os dados por limiares; transformações que preservam a ordem dos valores não mudam as divisões possíveis. Por isso os outliers também não precisam de tratamento. |
| Categóricos como códigos inteiros | `OperatingSystems`, `Browser`, `Region` e `TrafficType` já são inteiros; `Month` (em ordem de calendário) e `VisitorType` são convertidos em códigos. Com one-hot, o `TrafficType` viraria 20 colunas binárias, cada uma raramente escolhida para divisão. |
| `class_weight="balanced"` | Dá a cada classe um peso inversamente proporcional à sua frequência: errar uma sessão "comprou" pesa cerca de 5,4 vezes mais, sem alterar os dados. |

In [6]:
preprocessamento_rf = ColumnTransformer([
    ("codigos", OrdinalEncoder(categories=[ORDEM_MESES, ["New_Visitor", "Other", "Returning_Visitor"]]),
     ["Month", "VisitorType"]),
], remainder="passthrough", verbose_feature_names_out=False)   # demais atributos entram sem mudança

pipeline_rf = Pipeline([
    ("preprocessamento", preprocessamento_rf),
    ("modelo", RandomForestClassifier(class_weight="balanced", random_state=SEMENTE, n_jobs=-1)),
])
pipeline_rf

,steps,"[('preprocessamento', ...), ('modelo', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('codigos', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_we

### 3.3 Gradient Boosting (`HistGradientBoostingClassifier`)

| O que é feito | Justificativa |
|---|---|
| Manter todos os atributos, sem log e sem escala | Mesma lógica do Random Forest: o modelo é baseado em árvores. |
| Categóricos nativos (`categorical_features`) | O modelo trata cada categoria como categoria, sem impor ordem e sem one-hot. Os categóricos são apenas convertidos em códigos 0, 1, 2…, formato exigido por esse recurso; uma categoria nunca vista no treino vira valor ausente, que o modelo sabe tratar. |
| `class_weight="balanced"` | Mesmo motivo do Random Forest. |

In [7]:
preprocessamento_gb = ColumnTransformer([
    ("codigos", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), CATEGORICOS),
], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")

pipeline_gb = Pipeline([
    ("preprocessamento", preprocessamento_gb),
    ("modelo", HistGradientBoostingClassifier(categorical_features=CATEGORICOS,
                                              class_weight="balanced", random_state=SEMENTE)),
])
pipeline_gb

,steps,"[('preprocessamento', ...), ('modelo', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('codigos', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_we

### 3.4 MLP

| O que é feito | Justificativa |
|---|---|
| Remover `Administrative`, `Informational` e `ProductRelated` | Mesma redundância identificada no gráfico de correlações. |
| Padronização (z-score) nos atributos numéricos | Deixa os atributos centrados em 0 com desvio padrão 1. A rede aprende por gradiente, e entradas em escalas muito diferentes (`ProductRelated_Duration` chega a 63.973, `ExitRates` não passa de 0,2) deixam o treino instável e lento. |
| One-hot nos categóricos | A rede multiplica as entradas por pesos; usar os códigos diretamente faria o Browser 13 "valer mais" que o Browser 2. |
| Agrupar categorias com menos de 1% das sessões | Reduz o ruído e o número de entradas da rede. |
| Sobreamostragem aleatória da classe "comprou" (só no fold de treino) | O `MLPClassifier` do scikit-learn não aceita pesos por classe. |

In [8]:
preprocessamento_mlp = ColumnTransformer([
    ("zscore", StandardScaler(), DURACOES_E_PAGEVALUES + TAXAS),
    ("onehot", one_hot_com_raras(), CATEGORICOS),
    ("binarios", "passthrough", BINARIOS),
], remainder="drop", verbose_feature_names_out=False)

pipeline_mlp = Pipeline([
    ("preprocessamento", preprocessamento_mlp),
    ("balanceamento", RandomOverSampler(random_state=SEMENTE)),
    ("modelo", MLPClassifier(random_state=SEMENTE)),
])
pipeline_mlp

,steps,"[('preprocessamento', ...), ('balanceamento', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('zscore', ...), ('onehot', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more detai

## 4. Verificação em um fold do k-fold

Para conferir o que cada pipeline faz, aplica-se o pré-processamento ao **primeiro fold** de um
`StratifiedKFold` com 10 folds:

- as etapas são ajustadas **só no treino** e depois aplicadas ao teste;
- a sobreamostragem acontece **só no treino**: o teste mantém a proporção real das classes.

In [ ]:
PIPELINES = {
    "KNN": pipeline_knn,
    "Random Forest": pipeline_rf,
    "Gradient Boosting": pipeline_gb,
    "MLP": pipeline_mlp,
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEMENTE)
idx_treino, idx_teste = next(cv.split(X, y))
X_treino, X_teste = X.iloc[idx_treino], X.iloc[idx_teste]
y_treino, y_teste = y.iloc[idx_treino], y.iloc[idx_teste]

linhas = []
transformados = {}
ajustados = {}   # cópias ajustadas no fold; os pipelines originais continuam sem treinamento
for nome, pipe in PIPELINES.items():
    pipe = ajustados[nome] = clone(pipe)
    pre = pipe.named_steps["preprocessamento"]
    Xt_treino = pre.fit_transform(X_treino, y_treino)
    Xt_teste = pre.transform(X_teste)
    yt_treino = y_treino
    if "balanceamento" in pipe.named_steps:
        Xt_treino, yt_treino = pipe.named_steps["balanceamento"].fit_resample(Xt_treino, y_treino)
    transformados[nome] = pd.DataFrame(Xt_teste, columns=pre.get_feature_names_out())
    linhas.append({
        "algoritmo": nome,
        "colunas_entrada": X_treino.shape[1],
        "colunas_saida": Xt_teste.shape[1],
        "treino_antes": len(y_treino),
        "treino_depois": len(yt_treino),
        "pct_comprou_treino_antes": 100 * y_treino.mean(),
        "pct_comprou_treino_depois": 100 * np.mean(yt_treino),
        "pct_comprou_teste": 100 * y_teste.mean(),
    })
resumo = pd.DataFrame(linhas).set_index("algoritmo")
resumo.round(2)Error

,colunas_entrada,colunas_saida,treino_antes,treino_depois,pct_comprou_treino_antes,pct_comprou_treino_depois,pct_comprou_teste
algoritmo,,,,,,,
KNN,17,55,10984,18534,15.63,50.00,15.64
Random Forest,17,17,10984,10984,15.63,15.63,15.64
Gradient Boosting,17,17,10984,10984,15.63,15.63,15.64
MLP,17,55,10984,18534,15.63,50.00,15.64


Pesos que o `class_weight="balanced"` atribui a cada classe no Random Forest e no Gradient Boosting (calculados no fold de treino):

In [10]:
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_treino)
pd.DataFrame({"classe": ["não comprou (0)", "comprou (1)"], "peso": pesos,
              "peso_relativo": pesos / pesos[0]}).set_index("classe").round(3)

,peso,peso_relativo
classe,,
não comprou (0),0.593,1.000
comprou (1),3.199,5.397


Colunas geradas no KNN. As colunas `*_infrequent_sklearn` reúnem as categorias raras de cada atributo:

In [11]:
nomes_knn = ajustados["KNN"].named_steps["preprocessamento"].get_feature_names_out()
print(f"{len(nomes_knn)} colunas:")
print(", ".join(nomes_knn))
print()
print("Contagens removidas:", [c for c in REDUNDANTES if c in nomes_knn] or "nenhuma restante ✔")

55 colunas:
Administrative_Duration, Informational_Duration, ProductRelated_Duration, PageValues, BounceRates, ExitRates, SpecialDay, Month_Aug, Month_Dec, Month_Feb, Month_Jul, Month_Jun, Month_Mar, Month_May, Month_Nov, Month_Oct, Month_Sep, OperatingSystems_1, OperatingSystems_2, OperatingSystems_3, OperatingSystems_4, OperatingSystems_infrequent_sklearn, Browser_1, Browser_2, Browser_4, Browser_5, Browser_6, Browser_8, Browser_10, Browser_infrequent_sklearn, Region_1, Region_2, Region_3, Region_4, Region_5, Region_6, Region_7, Region_8, Region_9, TrafficType_1, TrafficType_2, TrafficType_3, TrafficType_4, TrafficType_5, TrafficType_6, TrafficType_8, TrafficType_10, TrafficType_11, TrafficType_13, TrafficType_20, TrafficType_infrequent_sklearn, VisitorType_New_Visitor, VisitorType_Returning_Visitor, VisitorType_infrequent_sklearn, Weekend

Contagens removidas: nenhuma restante ✔


In [12]:
onehot = ajustados["KNN"].named_steps["preprocessamento"].named_transformers_["onehot"]
agrupadas = pd.DataFrame([
    {"atributo": atributo, "categorias_agrupadas": ", ".join(map(str, cats)), "quantidade": len(cats)}
    for atributo, cats in zip(CATEGORICOS, onehot.infrequent_categories_) if cats is not None
]).set_index("atributo")
agrupadas

,categorias_agrupadas,quantidade
atributo,,
OperatingSystems,"5, 6, 7, 8",4
Browser,"3, 7, 9, 11, 12, 13",6
TrafficType,"7, 9, 12, 14, 15, 16, 18, 19",8
VisitorType,Other,1


Faixa dos valores após o pré-processamento (fold de **teste**) para KNN e MLP:

In [13]:
numericos = DURACOES_E_PAGEVALUES + TAXAS
faixas = pd.concat({
    "KNN (log + min-max)": transformados["KNN"][numericos].agg(["min", "median", "max"]).T,
    "MLP (z-score)": transformados["MLP"][numericos].agg(["mean", "std", "min", "max"]).T,
}, axis=1)
faixas.round(3)

KNN (log + min-max)               MLP (z-score)  \
                                        min median    max          mean   
Administrative_Duration                 0.0  0.295  0.958        -0.002   
Informational_Duration                  0.0  0.000  0.933        -0.018   
ProductRelated_Duration                 0.0  0.585  0.908         0.014   
PageValues                              0.0  0.000  0.951         0.043   
BounceRates                             0.0  0.013  1.000        -0.028   
ExitRates                               0.0  0.120  1.000        -0.036   
SpecialDay                              0.0  0.000  1.000         0.046   

                                               
                           std    min     max  
Administrative_Duration  0.982 -0.459  13.080  
Informational_Duration   0.858 -0.245  10.303  
ProductRelated_Duration  0.980 -0.626  11.358  
PageValues               1.175 -0.321  14.471  
BounceRates              0.935 -0.450   3.942  
ExitRates                0.949 -0.897   3.414  
SpecialDay               1.075 -0.308   4.740

Como o min-max e o z-score foram ajustados **só no treino**, o fold de teste não fica exatamente com as propriedades do treino: no MLP, a média não é exatamente 0 nem o desvio exatamente 1, e no KNN um valor de teste maior que o máximo do treino ficaria acima de 1. Isso é esperado e indica que o teste não influenciou o ajuste.

## 5. Salvar os pipelines

Os pipelines são salvos **sem treinamento** em `02_preprocessamento/pipelines/`. Na fase de classificação, basta
carregá-los, ajustar os hiperparâmetros do modelo e usar a validação cruzada:

```python
pipeline = joblib.load(DIR_PIPELINES / "knn.joblib")
pipeline.set_params(modelo__n_neighbors=15)
cross_validate(pipeline, X, y, cv=StratifiedKFold(10, shuffle=True, random_state=42), ...)
```

A cada fold, o `cross_validate` reajusta todas as etapas do pipeline só com o treino daquele fold.

In [14]:
ARQUIVOS = {
    "KNN": "knn.joblib",
    "Random Forest": "random_forest.joblib",
    "Gradient Boosting": "gradient_boosting.joblib",
    "MLP": "mlp.joblib",
}
for nome, arquivo in ARQUIVOS.items():
    joblib.dump(PIPELINES[nome], DIR_PIPELINES / arquivo)
    print(f"{nome:<18} -> {(DIR_PIPELINES / arquivo).relative_to(RAIZ)}")

KNN                -> 02_preprocessamento/pipelines/knn.joblib
Random Forest      -> 02_preprocessamento/pipelines/random_forest.joblib
Gradient Boosting  -> 02_preprocessamento/pipelines/gradient_boosting.joblib
MLP                -> 02_preprocessamento/pipelines/mlp.joblib
